# SFINCS Scenario Stats

> **Stage Contract**
>
> Requires: scenario folders and completed run_outputs for staged Event Catalog rows
>
> Produces: scenario statistics and evaluation products under data/sfincs/stats
>
> Next: publish results or iterate on source/scenario settings

In [ ]:
import sys
from pathlib import Path

notebook_root = Path.cwd().resolve()
location_root = notebook_root.parent
repo_root = location_root.parents[1]
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))
config_yaml = location_root / "config.yaml"

# Load notebook tools.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

from sfincs_runs.config import load_runtime
from sfincs_runs.scenarios import scenario_stats as stats

config, paths = load_runtime(config_yaml)
pd.set_option("display.max_columns", 80)


## Step 0: Load Stats Inputs

Stats are not raw SFINCS outputs alone. They need three folder families:

- `scenarios_root`: prepared SFINCS event folders with `sfincs.inp`, `sfincs.bzs`, and manifests.
- `storage_root`: persisted solver outputs, especially `sfincs_map.nc`.
- `design_outputs_root`: `design_events` outputs that explain the sampled peak, template, return period, and SLR scenario.

These paths come from `sfincs_runs/project.yaml` through `build_paths()`.

In [ ]:
settings = pd.Series(
    {
        "scenarios_root": str(paths["scenarios_root"]),
        "storage_root": str(paths["storage_root"]),
        "stats_root": str(paths["stats_root"]),
        "design_outputs_root": str(paths["design_outputs_root"]),
        "land_threshold_m": 0.0,
        "huthresh_m": 0.01,
        "impact_threshold_m": 0.10,
    },
    name="value",
)
settings

## Step 1

- Which SLR scenario generated this run?
- What return period was sampled?
- Which historical hydrograph template shaped the event?
- Does the SFINCS boundary file match the expected design-event peak?

In [ ]:
scenario_summary, scenario_rows = stats.load_scenario_build(paths["scenarios_root"])
design_rows, design_attrs = stats.load_design_events(scenario_summary)

display(pd.Series(scenario_summary, name="scenario_build"))
display(pd.Series(design_attrs, name="design_event_attrs"))

design_frame = pd.DataFrame.from_dict(design_rows, orient="index")
cols = [c for c in ["sample_rp_years", "peak_m", "absolute_peak_m", "template_id", "template_peak_time", "tail_morph_factor"] if c in design_frame]
design_frame[cols].head()

## Step 2: Pick Completed Events

A scenario folder is the preferred prepared input. A completed run also has `sfincs_map.nc` in the storage folder.

If `data/sfincs/scenarios` only contains a debug or partial staging set, but `data/sfincs/run_outputs` contains self-contained completed run folders, the next cell falls back to those run-output folders so the evaluation catalogue does not silently shrink.

In [ ]:
# Scenario folders are the normal evaluation unit. Run-output folders are a safe fallback
# when they carry the copied sfincs.inp, sfincs.bzs, forcing manifest, and map output.
event_inventory = stats.completed_event_inventory(paths["scenarios_root"], paths["storage_root"])
all_events = event_inventory["all_events"]
completed_events = event_inventory["completed_events"]

LIMIT = 12  # set to None for all completed events in the preview table
selected_events = completed_events[:LIMIT] if LIMIT is not None else completed_events

pd.Series(
    {
        "scenario_folder_count": len(event_inventory["scenario_events"]),
        "storage_run_output_count": len(event_inventory["storage_events"]),
        "scenario_completed_count": len(event_inventory["scenario_completed"]),
        "storage_completed_count": len(event_inventory["storage_completed"]),
        "event_source_root": str(event_inventory["event_source_root"]),
        "using_storage_outputs": bool(event_inventory["use_storage_outputs"]),
        "prepared_event_count": len(all_events),
        "completed_event_count": len(completed_events),
        "selected_event_count": len(selected_events),
        "first_selected_event": selected_events[0].name if selected_events else None,
    }
)

## Step 2b: Health Check All Runs

Before trusting any aggregate, confirm each event finished cleanly. This cell walks every event in `completed_events` and flags anomalies on three axes:

- `returncode` from `run_metadata.json` (should be 0).
- `sfincs_map.nc` file size (a normal run is tens of MB; near zero means the solver wrote no map output).
- finite-value fraction of the final-timestep `zs` slice (near zero means the solver wrote NaNs across the active grid).

Also surfaces per-event wall-clock duration so outliers — events much slower than the rest — are visible before any downstream analysis. The slow step is opening 500 NetCDFs once (~1 min on a warm cache).

In [ ]:
import json

health_rows = []
for d in completed_events:
    meta_path = paths["storage_root"] / d.name / "run_metadata.json"
    map_path = paths["storage_root"] / d.name / "sfincs_map.nc"
    rec = {"event_id": d.name, "open_error": ""}

    if meta_path.exists():
        meta = json.loads(meta_path.read_text())
        rec["returncode"] = int(meta.get("returncode", -1))
        rec["duration_min"] = float(meta.get("duration_sec", float("nan"))) / 60.0
    else:
        rec["returncode"] = None
        rec["duration_min"] = float("nan")

    rec["map_mb"] = map_path.stat().st_size / (1024 * 1024) if map_path.exists() else 0.0

    try:
        with xr.open_dataset(map_path, decode_times=False) as ds:
            zs_last = ds["zs"].isel(time=-1).values
            rec["zs_finite_frac"] = float(np.isfinite(zs_last).mean())
            rec["zs_max_last_m"] = float(np.nanmax(zs_last)) if np.any(np.isfinite(zs_last)) else float("nan")
            rec["n_timesteps"] = int(ds.sizes.get("time", 0))
    except Exception as exc:
        rec["zs_finite_frac"] = float("nan")
        rec["zs_max_last_m"] = float("nan")
        rec["n_timesteps"] = 0
        rec["open_error"] = str(exc)[:120]

    health_rows.append(rec)

health = pd.DataFrame(health_rows)

flags = {
    "bad returncode":        health["returncode"].fillna(-1) != 0,
    "tiny map (<1 MB)":      health["map_mb"] < 1.0,
    "open failed":           health["open_error"] != "",
    "empty zs (<5% finite)": health["zs_finite_frac"] < 0.05,
    "missing timesteps":     health["n_timesteps"] < 2,
}

print(f"events checked: {len(health)}\n")
for label, mask in flags.items():
    print(f"  {label:24s} {int(mask.sum())}")

print("\nduration (min):")
display(health["duration_min"].describe().to_frame().T)

flagged_mask = pd.Series(False, index=health.index)
for mask in flags.values():
    flagged_mask = flagged_mask | mask

if flagged_mask.any():
    print(f"\n{int(flagged_mask.sum())} anomalous events:")
    display(health.loc[flagged_mask].sort_values("event_id"))
else:
    print("\nAll events look clean.")

## Step 3: Inspect One Event End to End

For one event, compare the design-event forcing metadata with SFINCS results.

Important metric logic:
- `zs - zb` gives water depth.
- land cells are active cells with bed elevation above `land_threshold_m`.
- baseline wet land at `t0` is not counted as new impact.
- incremental flood depth measures added land depth above the `t0` baseline.
- impact extent uses the configured `impact_threshold_m`.

In [ ]:
if not selected_events:
    raise RuntimeError("No completed SFINCS events found. Run scenarios first.")

event_dir = selected_events[0]
row = stats.event_stats(
    event_dir,
    paths["storage_root"],
    settings["land_threshold_m"],
    settings["huthresh_m"],
    settings["impact_threshold_m"],
    scenario_summary,
    scenario_rows,
    design_rows,
    design_attrs,
)

focus = [
    "event_id", "design_scenario", "design_slr_offset_m", "sample_rp_years", "template_id",
    "driver_h_magnitude", "expected_bzs_peak_max_m", "bzs_peak_max_m",
    "peak_incremental_land_depth_m", "peak_incremental_flooded_area_km2",
    "longest_incremental_flood_duration_h", "area_incremental_flooded_ge_24h_km2",
]
pd.Series({k: row.get(k) for k in focus}, name=event_dir.name)

## Step 4: Visualize the One-Event Flood Signal

- left: max incremental land depth per cell.
- right: wet land cell count through time.

In [ ]:
inp = stats.parse_sfincs_inp(event_dir / "sfincs.inp")
with xr.open_dataset(row["map_path"], decode_times=False) as ds:
    zs = np.asarray(ds["zs"].values, np.float32)
    zb = np.asarray(ds["zb"].values, np.float32)
    active = np.asarray(ds["msk"].values, float) > 0
    hours, dt = stats.parse_time(ds, inp)

depth = np.where(active[None, :, :], zs - zb[None, :, :], np.nan)
land = active & np.isfinite(zb) & (zb > settings["land_threshold_m"])
baseline = np.where(land, np.maximum(depth[0], 0.0), np.nan)
incremental = np.where(land[None, :, :], np.maximum(depth - np.nan_to_num(baseline, nan=0.0)[None, :, :], 0.0), np.nan)
impact = np.isfinite(incremental) & (incremental > settings["impact_threshold_m"])

peak_map = np.full(land.shape, np.nan, float)
impacted_cells = np.any(impact, axis=0)
if np.any(impacted_cells):
    peak_map[impacted_cells] = np.nanmax(np.where(impact, incremental, np.nan)[:, impacted_cells], axis=0)
impact_counts = impact.sum(axis=(1, 2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
im = axes[0].imshow(peak_map, origin="lower", cmap="viridis")
axes[0].set_title(f"{event_dir.name}: peak incremental land depth [m]")
fig.colorbar(im, ax=axes[0], shrink=0.75)

axes[1].plot(hours, impact_counts, lw=1.5)
axes[1].set_title("Impacted land cells through time")
axes[1].set_xlabel("hour since start")
axes[1].set_ylabel("cell count")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 5: Build the Stats Table

In [ ]:
rows = []
for d in selected_events:
    rows.append(
        stats.event_stats(
            d,
            paths["storage_root"],
            settings["land_threshold_m"],
            settings["huthresh_m"],
            settings["impact_threshold_m"],
            scenario_summary,
            scenario_rows,
            design_rows,
            design_attrs,
        )
    )

df = pd.DataFrame(rows).sort_values("event_id").reset_index(drop=True)
display(df[[
    "event_id", "design_scenario", "design_slr_offset_m", "sample_rp_years",
    "expected_bzs_peak_max_m", "bzs_peak_max_m",
    "peak_incremental_land_depth_m", "peak_incremental_flooded_area_km2",
    "longest_incremental_flood_duration_h",
]].head())

df.describe(include="all").T.head(20)

## Step 6: Check Alignment Between Inputs and Flood Outputs

- boundary peak from `sfincs.bzs` should track the design-event expected boundary peak.
- return period and SLR scenario should be available beside flood metrics.
- flood extent/depth rankings should point back to event IDs for map inspection.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].scatter(df["expected_bzs_peak_max_m"], df["bzs_peak_max_m"], s=24, alpha=0.8)
axes[0].set_title("Expected vs written boundary peak")
axes[0].set_xlabel("expected bzs peak [m]")
axes[0].set_ylabel("written bzs peak [m]")
axes[0].grid(True, alpha=0.3)

axes[1].scatter(df["sample_rp_years"], df["peak_incremental_land_depth_m"], s=24, alpha=0.8)
axes[1].set_xscale("log")
axes[1].set_title("Return period vs flood depth")
axes[1].set_xlabel("sample return period [years]")
axes[1].set_ylabel("peak incremental land depth [m]")
axes[1].grid(True, alpha=0.3)

axes[2].scatter(df["peak_incremental_land_depth_m"], df["peak_incremental_flooded_area_km2"], s=24, alpha=0.8)
axes[2].set_title("Depth vs extent")
axes[2].set_xlabel("peak incremental land depth [m]")
axes[2].set_ylabel("peak incremental flood extent [km^2]")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

df.sort_values("peak_incremental_land_depth_m", ascending=False)[["event_id", "design_scenario", "sample_rp_years", "peak_incremental_land_depth_m", "peak_incremental_flooded_area_km2"]].head(10)

## Step 7: Write Notebook Outputs

In [ ]:
outdir = paths["stats_root"] / "notebook"
outdir.mkdir(parents=True, exist_ok=True)

csv_path = outdir / "scenario_stats_notebook.csv"
df.to_csv(csv_path, index=False)

metric_cols = [
    "zsini_m", "baseline_t0_flooded_area_km2", "peak_incremental_land_depth_m",
    "peak_incremental_flooded_area_km2", "anytime_incremental_flooded_area_km2",
    "longest_incremental_flood_duration_h", "mean_incremental_flood_duration_h",
    "area_incremental_flooded_ge_24h_km2",
]
summary = {
    "event_count": int(len(df)),
    "design_outputs_root": scenario_summary.get("design_outputs_root"),
    "design_scenarios": sorted(str(x) for x in df["design_scenario"].dropna().unique()),
    "design_slr_offsets_m": sorted(float(x) for x in pd.to_numeric(df["design_slr_offset_m"], errors="coerce").dropna().unique()),
    "metric_summaries": {c: stats.series_summary(df, c) for c in metric_cols},
}

(outdir / "summary_notebook.json").write_text(__import__("json").dumps(summary, indent=2), encoding="utf-8")
pd.Series({"csv": str(csv_path), "summary": str(outdir / "summary_notebook.json")})

## Step 8: Export Run-Output Catalogue

In [ ]:
# Catalogue join: completed run_outputs -> Event Catalog descriptors -> flood outcome metrics.
import json
import numpy as np

catalogue_events = completed_events
run_index = pd.DataFrame(
    {
        "event_id": [d.name for d in catalogue_events],
        "scenario_dir": [str(d) for d in catalogue_events],
        "run_output_dir": [str(paths["storage_root"] / d.name) for d in catalogue_events],
    }
)

catalogue_stats_path = paths["stats_root"] / "scenario_stats.csv"
if catalogue_stats_path.exists():
    catalogue_outcomes = pd.read_csv(catalogue_stats_path)
    missing_outcome_ids = sorted(set(run_index["event_id"]) - set(catalogue_outcomes["event_id"].astype(str)))
else:
    catalogue_outcomes = pd.DataFrame()
    missing_outcome_ids = run_index["event_id"].tolist()

if missing_outcome_ids:
    print(f"Building catalogue outcome metrics for {len(missing_outcome_ids)} completed events...")
    missing_outcome_set = set(missing_outcome_ids)
    fresh_rows = []
    for i, d in enumerate([d for d in catalogue_events if d.name in missing_outcome_set], start=1):
        fresh_rows.append(
            stats.event_stats(
                d,
                paths["storage_root"],
                settings["land_threshold_m"],
                settings["huthresh_m"],
                settings["impact_threshold_m"],
                scenario_summary,
                scenario_rows,
                design_rows,
                design_attrs,
            )
        )
        if i % 50 == 0 or i == len(missing_outcome_ids):
            print(f"  {i}/{len(missing_outcome_ids)} outcome rows")
    fresh_outcomes = pd.DataFrame(fresh_rows)
    catalogue_outcomes = pd.concat([catalogue_outcomes, fresh_outcomes], ignore_index=True)

catalogue_outcomes = catalogue_outcomes[catalogue_outcomes["event_id"].astype(str).isin(set(run_index["event_id"]))]

# Event Catalog rows are the physical forcing contract used to stage these SFINCS runs.
event_catalog_path = paths["design_outputs_root"] / "catalog" / "event_catalog.csv"
event_catalog = pd.read_csv(event_catalog_path).rename(
    columns={
        "event_reference_time": "catalog_event_reference_time",
        "event_drivers": "driver_combo",
    }
)
event_catalog["event_catalog_csv"] = str(event_catalog_path)

# Driver magnitudes follow the Marshfield Event Catalog schema: coastal peak [m] plus AORC 72h mean rainfall [mm].
rainfall_table_cache = {}
rainfall_metrics = []
for _, event_row in event_catalog.iterrows():
    rainfall_file = event_row.get("rainfall_member_file")
    rainfall_time = pd.to_datetime(event_row.get("rainfall_member_time"), errors="coerce")
    rainfall_metric = np.nan
    if pd.notna(rainfall_time) and isinstance(rainfall_file, str) and rainfall_file.strip():
        rainfall_path = Path(rainfall_file)
        if rainfall_path.exists():
            if rainfall_path not in rainfall_table_cache:
                rainfall_table_cache[rainfall_path] = pd.read_csv(rainfall_path)
            rainfall_table = rainfall_table_cache[rainfall_path]
            matched = rainfall_table[pd.to_datetime(rainfall_table["storm_date"], errors="coerce") == rainfall_time]
            if not matched.empty and "mean" in matched:
                rainfall_metric = float(matched.iloc[0]["mean"])
    rainfall_metrics.append(rainfall_metric)
event_catalog["rainfall_metric_mm"] = rainfall_metrics
event_catalog["driver_h_magnitude"] = pd.to_numeric(event_catalog.get("coastal_absolute_peak_m"), errors="coerce")
event_catalog["driver_p_magnitude"] = pd.to_numeric(event_catalog["rainfall_metric_mm"], errors="coerce")

# The Event Catalog carries browsing labels for every staged run; the stress/training table fills older synthetic rows.
stress_catalog_path = paths["design_outputs_root"] / "catalog" / "resilience_stress_training_catalog.csv"
stress_cols = [
    "event_id", "event_origin", "catalog_role", "storm_type", "event_set",
    "selection_role", "selection_reason", "benchmark_return_period_years",
    "compound_pairing_policy", "compound_pairing_role", "scenario_timing_edge_case",
]
stress_cols = [c for c in stress_cols if c in pd.read_csv(stress_catalog_path, nrows=0).columns]
stress_catalog = pd.read_csv(stress_catalog_path, usecols=stress_cols)
stress_catalog = stress_catalog.rename(columns={c: f"{c}_stress" for c in stress_catalog.columns if c != "event_id"})
stress_catalog["classification_catalog_csv"] = str(stress_catalog_path)

manifest_rows = []
for d in catalogue_events:
    manifest_path = paths["storage_root"] / d.name / "forcing_manifest.json"
    manifest = json.loads(manifest_path.read_text())
    manifest_rows.append(
        {
            "event_id": d.name,
            "run_event_reference_time": manifest.get("event_reference_time"),
            "run_start": manifest.get("run_start"),
            "run_stop": manifest.get("run_stop"),
            "run_duration_hours": manifest.get("run_duration_hours"),
            "model_t_start": manifest.get("t_start"),
            "model_t_stop": manifest.get("t_stop"),
            "timing_policy": manifest.get("timing_policy"),
            "forcing_variable": manifest.get("forcing_variable"),
            "expected_has_precip": manifest.get("expected_has_precip"),
            "expected_has_waves": manifest.get("expected_has_waves"),
            "initial_soil_moisture_fraction": manifest.get("initial_soil_moisture_fraction"),
            "run_soil_moisture_member_file": manifest.get("soil_moisture_member_file"),
            "run_soil_moisture_member_id": manifest.get("soil_moisture_member_id"),
            "run_soil_moisture_member_time": manifest.get("soil_moisture_member_time"),
            "soil_moisture_included_in_run": (
                manifest.get("initial_soil_moisture_fraction") is not None
                or any(str(manifest.get(k) or "").strip() for k in ["soil_moisture_member_file", "soil_moisture_member_id", "soil_moisture_member_time"])
            ),
            "driver_windows": json.dumps(manifest.get("driver_windows", [])),
            "prepared_precip": manifest.get("prepared_precip"),
            "rainfall_source_nc": manifest.get("rainfall_source_nc"),
            "surge_dataset": manifest.get("surge_dataset"),
            "base_model_root": manifest.get("base_model_root"),
        }
    )
manifest_frame = pd.DataFrame(manifest_rows)

outcome_cols = [
    "event_id", "map_path", "design_scenario", "design_slr_offset_m",
    "expected_bzs_peak_max_m", "bzs_peak_max_m",
    "peak_incremental_land_depth_m", "peak_incremental_flooded_area_km2",
    "anytime_incremental_flooded_area_km2", "peak_newly_flooded_area_km2",
    "anytime_newly_flooded_area_km2", "longest_incremental_flood_duration_h",
    "mean_incremental_flood_duration_h", "area_incremental_flooded_ge_24h_km2",
    "cumulative_incremental_flooded_area_km2h", "available_time_steps",
    "last_output_hour", "active_cells", "land_t0_cells", "cell_area_m2",
]
outcome_cols = [c for c in outcome_cols if c in catalogue_outcomes.columns]

health_cols = [
    "event_id", "returncode", "duration_min", "map_mb", "n_timesteps",
    "zs_finite_frac", "zs_max_last_m", "open_error",
]
health_cols = [c for c in health_cols if c in health.columns]

catalogue = (
    run_index
    .merge(event_catalog, on="event_id", how="left")
    .merge(stress_catalog, on="event_id", how="left")
    .merge(manifest_frame, on="event_id", how="left")
    .merge(catalogue_outcomes[outcome_cols], on="event_id", how="left")
    .merge(health[health_cols], on="event_id", how="left")
)

label_cols = [
    "event_origin", "catalog_role", "storm_type", "event_set",
    "selection_role", "selection_reason", "benchmark_return_period_years",
    "compound_pairing_policy", "compound_pairing_role", "scenario_timing_edge_case",
]
for col in label_cols:
    stress_col = f"{col}_stress"
    if col not in catalogue and stress_col in catalogue:
        catalogue[col] = catalogue[stress_col]
    elif stress_col in catalogue:
        catalogue[col] = catalogue[col].combine_first(catalogue[stress_col])

soil_cols = ["soil_moisture_source", "soil_moisture_member_id", "soil_moisture_member_time"]
for col in soil_cols:
    if col not in catalogue:
        catalogue[col] = ""
    catalogue[col] = catalogue[col].astype("object")
soil_absent = ~catalogue["soil_moisture_included_in_run"].fillna(False).astype(bool)
catalogue.loc[soil_absent, soil_cols] = "not_staged"

for col in ["event_origin", "storm_type", "severity_band", "driver_combo"]:
    if col not in catalogue:
        catalogue[col] = "unresolved"
    catalogue[col] = catalogue[col].fillna("unresolved").astype(str)

origin_group = catalogue["event_origin"].copy()
origin_group[origin_group.str.contains("historical", case=False, na=False)] = "historical"
origin_group[origin_group.str.contains("synthetic", case=False, na=False)] = "synthetic"
origin_group[origin_group.isin(["", "nan", "unresolved"])] = "unresolved"
catalogue.insert(0, "event_origin_group", origin_group)
catalogue.insert(
    1,
    "catalogue_section",
    catalogue["event_origin_group"] + " / " + catalogue["storm_type"] + " / " + catalogue["severity_band"],
)

severity_rank = {"mild": 0, "common": 1, "significant": 2, "rare": 3, "extreme": 4}
catalogue["_severity_rank"] = catalogue["severity_band"].map(severity_rank).fillna(99)
catalogue["_sample_rp_sort"] = pd.to_numeric(catalogue.get("sample_rp_years"), errors="coerce")
catalogue["_depth_sort"] = pd.to_numeric(catalogue.get("peak_incremental_land_depth_m"), errors="coerce")
catalogue = catalogue.sort_values(
    ["event_origin_group", "event_origin", "storm_type", "_severity_rank", "_sample_rp_sort", "_depth_sort", "event_id"],
    ascending=[True, True, True, True, True, False, True],
).drop(columns=["_severity_rank", "_sample_rp_sort", "_depth_sort"])

front_cols = [
    "event_origin_group", "catalogue_section", "event_origin", "storm_type", "severity_band",
    "event_id", "event_set", "catalog_role", "selection_role", "selection_reason",
    "sample_rp_years", "sampling_scheme", "sampling_region", "sampling_weight", "probability_weight",
    "driver_combo", "forcing_pairing_policy", "infiltration_treatment",
    "catalog_event_reference_time", "run_event_reference_time", "run_start", "run_stop", "run_duration_hours",
    "coastal_source", "coastal_member_id", "coastal_template_peak_time", "coastal_peak_m",
    "coastal_absolute_peak_m", "coastal_analog_id", "coastal_analog_peak_time",
    "coastal_water_level_scale_factor", "snapwave_source", "snapwave_member_id",
    "snapwave_valid_start_time", "snapwave_valid_end_time", "snapwave_pairing_policy",
    "rainfall_source", "rainfall_member_id", "rainfall_member_time", "rainfall_metric_mm", "rainfall_scale_factor",
    "rainfall_pairing_policy", "rainfall_pairing_reference_time", "rainfall_pairing_lag_hours",
    "soil_moisture_included_in_run", "soil_moisture_source", "soil_moisture_member_id", "soil_moisture_member_time",
    "driver_h_magnitude", "driver_p_magnitude", "expected_has_precip", "expected_has_waves",
    "initial_soil_moisture_fraction", "expected_bzs_peak_max_m", "bzs_peak_max_m",
    "peak_incremental_land_depth_m", "peak_incremental_flooded_area_km2",
    "anytime_incremental_flooded_area_km2", "peak_newly_flooded_area_km2",
    "anytime_newly_flooded_area_km2", "longest_incremental_flood_duration_h",
    "mean_incremental_flood_duration_h", "area_incremental_flooded_ge_24h_km2",
    "cumulative_incremental_flooded_area_km2h", "returncode", "duration_min", "map_mb", "n_timesteps",
    "available_time_steps", "last_output_hour", "run_output_dir", "scenario_dir", "map_path",
    "prepared_precip", "rainfall_source_nc", "surge_dataset", "base_model_root",
    "event_catalog_csv", "classification_catalog_csv",
]
front_cols = [c for c in front_cols if c in catalogue.columns]
catalogue = catalogue[front_cols + [c for c in catalogue.columns if c not in front_cols]]

catalogue_path = outdir / "flood_event_outcome_catalogue.csv"
catalogue.to_csv(catalogue_path, index=False)

display(catalogue.groupby(["event_origin_group", "storm_type", "severity_band"], dropna=False).size().rename("event_count").reset_index())
pd.Series({"catalogue_csv": str(catalogue_path), "events": int(len(catalogue))})


## Step 9: Return-Period Flood Diagnostics

Read the exported event-outcome catalogue back as the analysis unit: one completed event per row, carrying the sampled joint return period, driver magnitudes, storm class, and SFINCS flood-response metrics.

In [ ]:
# Flood-engineering diagnostics: exported catalogue -> RP bands, storm classes, and response ranks.
flood_catalogue_path = outdir / "flood_event_outcome_catalogue.csv"
flood = pd.read_csv(flood_catalogue_path)

numeric_flood_cols = [
    "sample_rp_years", "coastal_absolute_peak_m", "rainfall_metric_mm", "bzs_peak_max_m",
    "peak_incremental_land_depth_m", "peak_incremental_flooded_area_km2",
    "anytime_incremental_flooded_area_km2", "longest_incremental_flood_duration_h",
    "mean_incremental_flood_duration_h", "cumulative_incremental_flooded_area_km2h",
]
for col in numeric_flood_cols:
    flood[col] = pd.to_numeric(flood[col], errors="coerce")

# Return-period bins match common flood-risk communication thresholds: frequent, 10%, 2%, 1%, and 0.2% annual chances.
rp_breaks = [0, 2, 10, 50, 100, 500, np.inf]
rp_labels = ["<2 yr", "2-10 yr", "10-50 yr", "50-100 yr", "100-500 yr", ">500 yr"]
flood["rp_band"] = pd.cut(flood["sample_rp_years"], bins=rp_breaks, labels=rp_labels, right=False)
flood["has_incremental_flood"] = flood["anytime_incremental_flooded_area_km2"] > 0
p90 = lambda values: values.quantile(0.90)

# Band summary connects the sampled RP axis to depth, footprint, and duration consequences.
rp_band_stats = (
    flood.groupby(["rp_band", "severity_band"], observed=True)
    .agg(
        event_count=("event_id", "nunique"),
        median_rp_years=("sample_rp_years", "median"),
        median_peak_boundary_m=("bzs_peak_max_m", "median"),
        median_rainfall_mm=("rainfall_metric_mm", "median"),
        flood_hit_rate=("has_incremental_flood", "mean"),
        median_peak_depth_m=("peak_incremental_land_depth_m", "median"),
        p90_peak_depth_m=("peak_incremental_land_depth_m", p90),
        median_peak_area_km2=("peak_incremental_flooded_area_km2", "median"),
        p90_anytime_area_km2=("anytime_incremental_flooded_area_km2", p90),
        p90_longest_duration_h=("longest_incremental_flood_duration_h", p90),
    )
    .reset_index()
)

# Storm-type summary shows whether the flood response is controlled more by coastal-heavy or rainfall-heavy event families.
storm_type_stats = (
    flood.groupby(["storm_type", "severity_band"], observed=True)
    .agg(
        event_count=("event_id", "nunique"),
        median_rp_years=("sample_rp_years", "median"),
        median_peak_boundary_m=("bzs_peak_max_m", "median"),
        median_rainfall_mm=("rainfall_metric_mm", "median"),
        median_peak_area_km2=("peak_incremental_flooded_area_km2", "median"),
        p90_anytime_area_km2=("anytime_incremental_flooded_area_km2", p90),
        p90_longest_duration_h=("longest_incremental_flood_duration_h", p90),
    )
    .reset_index()
)

top_flood_events = flood.sort_values(
    ["anytime_incremental_flooded_area_km2", "peak_incremental_land_depth_m"],
    ascending=False,
)[[
    "event_id", "storm_type", "severity_band", "sample_rp_years",
    "coastal_absolute_peak_m", "rainfall_metric_mm", "bzs_peak_max_m",
    "peak_incremental_land_depth_m", "peak_incremental_flooded_area_km2",
    "anytime_incremental_flooded_area_km2", "longest_incremental_flood_duration_h",
]].head(15)

display(pd.Series({
    "catalogue_csv": str(flood_catalogue_path),
    "events": int(len(flood)),
    "storm_types": ", ".join(sorted(flood["storm_type"].dropna().unique())),
    "max_anytime_incremental_area_km2": flood["anytime_incremental_flooded_area_km2"].max(),
    "max_peak_incremental_depth_m": flood["peak_incremental_land_depth_m"].max(),
}, name="flood_catalogue_scope"))
display(rp_band_stats.round(3))
display(storm_type_stats.round(3))
display(top_flood_events.round(3))

severity_order = ["mild", "common", "significant", "rare", "extreme"]
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

for severity_band, group in flood.groupby("severity_band", sort=False):
    axes[0].scatter(
        group["sample_rp_years"], group["anytime_incremental_flooded_area_km2"],
        s=28, alpha=0.70, label=severity_band,
    )
axes[0].set_xscale("log")
axes[0].set_xlabel("sampled joint return period (years)")
axes[0].set_ylabel("anytime incremental flooded area (km2)")
axes[0].legend(title="severity", fontsize=8)

axes[1].boxplot(
    [flood.loc[flood["severity_band"] == band, "peak_incremental_flooded_area_km2"].dropna() for band in severity_order],
    tick_labels=severity_order,
    showfliers=False,
)
axes[1].set_ylabel("peak incremental flooded area (km2)")
axes[1].tick_params(axis="x", rotation=25)

for storm_type, group in flood.groupby("storm_type", sort=True):
    marker_size = np.clip(group["anytime_incremental_flooded_area_km2"] * 2.0, 12, 90)
    axes[2].scatter(
        group["rainfall_metric_mm"], group["coastal_absolute_peak_m"],
        s=marker_size, alpha=0.55, label=storm_type,
    )
axes[2].set_xlabel("72h mean rainfall (mm)")
axes[2].set_ylabel("coastal absolute peak (m)")
axes[2].legend(title="storm type", fontsize=8)

plt.tight_layout()
plt.show()
